# Prompt Registration
Register or update the system prompt in Vertex AI Prompt Management.
Run this notebook once to create the prompt, then copy the `prompt_id` into `gcp_rag.ipynb`.

In [1]:
# !gcloud auth application-default login

In [2]:
import google.auth
import vertexai
from vertexai import types, generative_models

_, project_id = google.auth.default()
LOCATION = "us-central1"

# Instantiate the Agent Platform client (new SDK)
client = vertexai.Client(project=project_id, location=LOCATION)
print(f"Project ID : {project_id}")
print(f"Client     : {client}")


Project ID : dev-mq-tech-transfer
Client     : <vertexai._genai.client.Client object at 0x0000026573D7BD40>


In [3]:
PROMPT_DISPLAY_NAME = "hello-world-prompt"
MODEL_NAME          = "gemini-2.5-flash"

# Content/Part live in google.genai.types, not vertexai.types
from google.genai import types as genai_types

# Version 1: simple hello world
_SYSTEM_INSTRUCTION_V1 = (
    "You are a friendly assistant. "
    "When greeted, respond with 'Hello, World!' and briefly introduce yourself."
)

# Version 2: enthusiastic hello world
_SYSTEM_INSTRUCTION_V2 = (
    "You are an enthusiastic and cheerful assistant. "
    "When greeted, respond with 'Hello, World!' using excitement and emojis, "
    "then warmly invite the user to ask you anything."
)

def make_prompt(system_instruction):
    return types.Prompt(
        prompt_data=types.PromptData(
            model=MODEL_NAME,
            system_instruction=genai_types.Content(
                role="user",
                parts=[genai_types.Part(text=system_instruction)],
            ),
            contents=[
                genai_types.Content(
                    role="user",
                    parts=[genai_types.Part(text="Hello!")],
                )
            ],
        ),
    )

print("Prompt templates defined (V1 and V2).")


Prompt templates defined (V1 and V2).


In [4]:
from google.genai import types as genai_types

KMS_KEY_NAME = (
    "projects/dev-mq-tech-transfer/locations/us-central1"
    "/keyRings/poc/cryptoKeys/poc"
)

_enc_spec = genai_types.EncryptionSpec(kms_key_name=KMS_KEY_NAME)
_create_cfg = types.CreatePromptVersionConfig(encryption_spec=_enc_spec)

# Register Version 1: create the prompt resource (with first version)
saved_v1 = client.prompts.create_version(
    prompt=make_prompt(_SYSTEM_INSTRUCTION_V1),
    config=_create_cfg,
)
REGISTERED_PROMPT_ID = saved_v1.prompt_id
print(f"Version 1 created  →  prompt_id: {REGISTERED_PROMPT_ID}  |  version_id: {saved_v1.version_id}")

# Register Version 2: add a second version to the same prompt resource via update()
saved_v2 = client.prompts.update(
    prompt_id=REGISTERED_PROMPT_ID,
    prompt=make_prompt(_SYSTEM_INSTRUCTION_V2),
)
print(f"Version 2 added    →  prompt_id: {saved_v2.prompt_id}  |  version_id: {saved_v2.version_id}")

print(f"\nView at: https://console.cloud.google.com/vertex-ai/generative/language/prompts?project={project_id}")


Version 1 created  →  prompt_id: 527730671838298112  |  version_id: 1


Version 2 added    →  prompt_id: 527730671838298112  |  version_id: 2

View at: https://console.cloud.google.com/vertex-ai/generative/language/prompts?project=dev-mq-tech-transfer


## Get a Saved Prompt

Retrieves the latest version of a prompt by its ID.

In [5]:
# Get the latest version of the prompt
retrieved_prompt = client.prompts.get(prompt_id=REGISTERED_PROMPT_ID)

print(f"prompt_id  : {retrieved_prompt.prompt_id}")
print(f"model      : {retrieved_prompt.prompt_data.model}")
print(f"system_instruction ({len(str(retrieved_prompt.prompt_data.system_instruction))} chars)")


prompt_id  : 527730671838298112
model      : gemini-2.5-flash
system_instruction (205 chars)


## Get a Specific Prompt Version

Use `get_version` when you need to pin to a specific version rather than always using the latest.

In [6]:
# List all versions and print version_id + system instruction for each
versions_meta = list(client.prompts.list_versions(prompt_id=REGISTERED_PROMPT_ID))
print(f"Total versions registered: {len(versions_meta)}\n")

for v in versions_meta:
    vp = client.prompts.get_version(
        prompt_id=REGISTERED_PROMPT_ID,
        version_id=v.version_id,
    )
    sys_instr = vp.prompt_data.system_instruction
    # system_instruction is a genai_types.Content object with .parts list
    if sys_instr is None:
        text = "(none)"
    elif hasattr(sys_instr, "parts") and sys_instr.parts:
        text = " ".join(
            p.text for p in sys_instr.parts if hasattr(p, "text") and p.text
        )
    else:
        text = str(sys_instr)

    print(f"version_id : {v.version_id}")
    print(f"prompt     : {text}")
    print()


Total versions registered: 2



version_id : 1
prompt     : You are an enthusiastic and cheerful assistant. When greeted, respond with 'Hello, World!' using excitement and emojis, then warmly invite the user to ask you anything.



version_id : 2
prompt     : You are an enthusiastic and cheerful assistant. When greeted, respond with 'Hello, World!' using excitement and emojis, then warmly invite the user to ask you anything.



## List All Prompts in the Project

Lists every prompt resource saved under the current GCP project.

In [7]:
# List all prompt resources in the project
prompt_refs = list(client.prompts.list())
print(f"Total prompts in project: {len(prompt_refs)}\n")
for ref in prompt_refs:
    p = client.prompts.get(prompt_id=ref.prompt_id)
    print(f"  prompt_id : {ref.prompt_id}")
    print(f"  model     : {p.prompt_data.model}")
    print()


Total prompts in project: 12



  prompt_id : 527730671838298112
  model     : gemini-2.5-flash



  prompt_id : 6035633016112414720
  model     : gemini-2.5-flash



  prompt_id : 2368014069572567040
  model     : gemini-2.5-flash

  prompt_id : 4982353653261139968
  model     : projects/dev-mq-tech-transfer/locations/us-central1/publishers/google/models/gemini-2.5-flash



  prompt_id : 1651941728820658176
  model     : projects/dev-mq-tech-transfer/locations/us-central1/publishers/google/models/gemini-2.5-flash



...

  prompt_id : 2258379566143766528
  model     : projects/dev-mq-tech-transfer/locations/us-central1/publishers/google/models/gemini-2.5-flash



  prompt_id : 34727250129584128
  model     : projects/dev-mq-tech-transfer/locations/us-central1/publishers/google/models/gemini-2.5-flash



  prompt_id : 6646011503109472256
  model     : projects/dev-mq-tech-transfer/locations/us-central1/publishers/google/models/gemini-2.5-flash



  prompt_id : 454687915381882880
  model     : projects/dev-mq-tech-transfer/locations/us-central1/publishers/google/models/gemini-2.5-flash

  prompt_id : 7336188146004000768
  model     : projects/dev-mq-tech-transfer/locations/us-central1/publishers/google/models/gemini-2.5-flash



  prompt_id : 6136182529838809088
  model     : projects/dev-mq-tech-transfer/locations/global/publishers/google/models/gemini-2.5-flash-lite



## Restore a Prompt Version

Promotes an older version back to the latest. Useful for rolling back a bad prompt update.

In [8]:
# Restore a previous version as the latest
# Set RESTORE_VERSION_ID to the version you want to promote
RESTORE_VERSION_ID = "1"   # change to your target version

restored_prompt = client.prompts.restore_version(
    prompt_id=REGISTERED_PROMPT_ID,
    version_id=RESTORE_VERSION_ID,
)
print(f"Version {RESTORE_VERSION_ID} restored as latest for prompt {restored_prompt.prompt_id}")


Version 1 restored as latest for prompt 527730671838298112


## Delete a Version / Delete a Prompt

> ⚠️ `delete` removes the entire prompt resource and **all** its versions permanently. `delete_version` removes only one version. Both are commented out below — uncomment carefully.

In [9]:
DELETE_VERSION_ID = "1"   # version to remove

# Delete a single version (keeps the prompt resource and other versions)
# client.prompts.delete_version(
#     prompt_id=REGISTERED_PROMPT_ID,
#     version_id=DELETE_VERSION_ID,
# )
# print(f"Version {DELETE_VERSION_ID} deleted from prompt {REGISTERED_PROMPT_ID}")

# Delete the entire prompt resource and ALL its versions — irreversible
# client.prompts.delete(prompt_id=REGISTERED_PROMPT_ID)
# print(f"Prompt {REGISTERED_PROMPT_ID} and all versions deleted")


## Hello World — verify the prompt loads and the model responds

In [10]:
# Load the saved prompt back from Vertex AI
retrieved = client.prompts.get(prompt_id=REGISTERED_PROMPT_ID)
print(f"Loaded prompt  : {retrieved.prompt_id}")
print(f"Model          : {retrieved.prompt_data.model}")

# Extract the system instruction text from the Content object
sys_instr = retrieved.prompt_data.system_instruction
if sys_instr and hasattr(sys_instr, "parts") and sys_instr.parts:
    sys_text = " ".join(p.text for p in sys_instr.parts if hasattr(p, "text") and p.text)
else:
    sys_text = str(sys_instr) if sys_instr else None

# Run a simple hello-world generation using the stored system instruction text
model = generative_models.GenerativeModel(
    model_name=retrieved.prompt_data.model,
    system_instruction=sys_text,
)

response = model.generate_content("Hello! Who are you and what can you help me with?")
print("\n--- Model response ---")
print(response.text)


Loaded prompt  : 527730671838298112
Model          : gemini-2.5-flash


C:\Users\L137844\Downloads\gap_assesment\tredence_gap_assessment\.venv\Lib\site-packages\vertexai\generative_models\_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()



--- Model response ---
Hello, World!

I am a friendly AI assistant, and I'm here to help you with a wide range of tasks. I can answer questions, provide information, brainstorm ideas, help with writing, summarize texts, and much more. Just let me know what you need!
